In [1]:
# ============================================================
# Notebook 24
# 24_bridge_main_source_aware_and_diagnostic_AURORA.ipynb
# Bridge diagnostic between main source-aware AURORA series
# and standalone diagnostic AURORA reconstruction
#
# Purpose:
#   Address reviewer concern that the main AURORA-vs-ROMA
#   comparison series differs from the diagnostic AURORA
#   mechanism-attribution series.
#
# Outputs:
#   - table_S32_main_vs_diagnostic_AURORA_bridge.csv
#   - table_S32_main_vs_diagnostic_AURORA_bridge_rounded.csv
#   - main_vs_diagnostic_AURORA_bridge_daily.csv
#   - NOTEBOOK24_bridge_validation_report_*.json
#
# Research diagnostics only. Not financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import re
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive mount skipped or already mounted:", repr(e))

import numpy as np
import pandas as pd

# ============================================================
# 0. User settings
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PROJECT_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / PROJECT_CODE

TABLE_DIR = OUTPUT_ROOT / "tables"
DIAG_DIR = OUTPUT_ROOT / "diagnostics"
REPORT_DIR = OUTPUT_ROOT / "reports"

for d in [TABLE_DIR, DIAG_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

STRICT_START = pd.Timestamp("2024-11-27")
STRICT_END = pd.Timestamp("2026-03-25")
ANNUALIZATION_DAYS = 252
CASH_CAP = 0.60

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

RUN_ROOT = OUTPUT_ROOT / "bridge_main_vs_diagnostic_AURORA" / f"run_{RUN_ID}"
RUN_TABLE_DIR = RUN_ROOT / "tables"
RUN_DIAG_DIR = RUN_ROOT / "diagnostics"
RUN_REPORT_DIR = RUN_ROOT / "reports"

for d in [RUN_ROOT, RUN_TABLE_DIR, RUN_DIAG_DIR, RUN_REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("Notebook 24: bridge main source-aware AURORA vs diagnostic AURORA")
print("RUN_ID:", RUN_ID)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("=" * 100)

# ============================================================
# 1. General utilities
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    path = Path(path)
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def normalize_name(x):
    return re.sub(r"[^A-Za-z0-9]+", "", str(x)).lower()

def looks_like_date_series(s):
    parsed = pd.to_datetime(s, errors="coerce")
    if not isinstance(parsed, pd.Series):
        parsed = pd.Series(parsed)
    if len(parsed) == 0:
        return False, parsed
    if parsed.notna().mean() < 0.50:
        return False, parsed
    years = parsed.dt.year
    if years.between(1990, 2035).mean() < 0.50:
        return False, parsed
    return True, parsed

def set_datetime_index_flex(df):
    df = df.copy()

    if isinstance(df.index, pd.DatetimeIndex):
        years = pd.Series(df.index.year)
        if years.between(1990, 2035).mean() > 0.50:
            df.index = pd.to_datetime(df.index)
            df.index.name = "date"
            return df.sort_index()

    preferred = [
        "date", "Date", "DATE", "datetime", "Datetime",
        "timestamp", "Timestamp", "Unnamed: 0", "index", "Index"
    ]
    candidate_cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]

    for c in candidate_cols:
        try:
            ok, parsed = looks_like_date_series(df[c])
            if ok:
                df = df.drop(columns=[c])
                df.index = pd.to_datetime(parsed)
                df.index.name = "date"
                return df[~df.index.isna()].sort_index()
        except Exception:
            pass

    idx_series = pd.Series(df.index)
    ok, parsed_idx = looks_like_date_series(idx_series)
    if ok:
        df.index = pd.to_datetime(parsed_idx.values)
        df.index.name = "date"
        return df[~df.index.isna()].sort_index()

    return df

def read_table_auto(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, low_memory=False)
    elif path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    return set_datetime_index_flex(df)

def find_files(patterns, roots, max_files=None):
    if isinstance(patterns, str):
        patterns = [patterns]
    if isinstance(roots, (str, Path)):
        roots = [roots]

    out = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            out.extend(root.rglob(pat))

    out = sorted(
        set([p for p in out if p.exists() and p.is_file()]),
        key=lambda p: (p.stat().st_mtime, p.as_posix()),
        reverse=True,
    )
    if max_files is not None:
        return out[:max_files]
    return out

def score_path_for_main_source_aware_return_matrix(path):
    text = path.as_posix().lower()
    name = path.name.lower()
    score = 0

    if "source" in text and "aware" in text:
        score += 10
    if "13b" in text or "notebook13b" in text:
        score += 8
    if "return_matrix" in name or "returnmatrix" in name:
        score += 8
    if "strict" in text or "aligned" in text:
        score += 4
    if "aurora" in text:
        score += 2
    if "archive" in text or "old" in text or "not_used" in text:
        score -= 10
    if "canonical_passive" in text:
        score -= 6
    if "equivalence_lambda_cash_sensitivity" in text:
        score -= 6
    if "aurora_exposure_matched_lambda" in text:
        score -= 6

    return score

def score_path_for_diagnostic_notebook18_returns(path):
    text = path.as_posix().lower()
    name = path.name.lower()
    score = 0

    if "all_notebook18_returns" in name:
        score += 20
    if "aurora_exposure_matched_lambda_reduced_universe" in text:
        score += 10
    if "returns" in text:
        score += 5
    if "archive" in text or "old" in text or "not_used" in text:
        score -= 10

    return score

def score_path_for_diagnostic_notebook18_weights(path):
    text = path.as_posix().lower()
    name = path.name.lower()
    score = 0

    if "all_notebook18_weights" in name:
        score += 20
    if "aurora_exposure_matched_lambda_reduced_universe" in text:
        score += 10
    if "weights" in text:
        score += 5
    if "archive" in text or "old" in text or "not_used" in text:
        score -= 10

    return score

def score_path_for_main_weights(path):
    text = path.as_posix().lower()
    name = path.name.lower()
    score = 0

    if "weight" in name or "weights" in name:
        score += 5
    if "source" in text and "aware" in text:
        score += 8
    if "aurora" in text:
        score += 4
    if "13b" in text or "notebook13b" in text:
        score += 6
    if "uamb" in normalize_name(name) or "uamvb" in normalize_name(name):
        score += 6
    if "notebook18" in text or "aurora_exposure_matched_lambda" in text:
        score -= 8
    if "archive" in text or "old" in text or "not_used" in text:
        score -= 10

    return score

def select_best_file(candidates, score_func, label, min_score=None):
    scored = sorted([(score_func(p), p) for p in candidates], key=lambda x: (x[0], x[1].as_posix()), reverse=True)

    print("\n" + "-" * 100)
    print(label)
    print("-" * 100)

    for score, p in scored[:15]:
        print(f"{score:>4}  {p}")

    if not scored:
        print("No candidates found.")
        return None

    best_score, best_path = scored[0]

    if min_score is not None and best_score < min_score:
        print(f"Best score {best_score} below minimum {min_score}. Returning None.")
        return None

    print("Selected:", best_path)
    return best_path

# ============================================================
# 2. Performance utilities
# ============================================================

def performance_metrics_from_returns(r):
    r = pd.Series(r).dropna().astype(float)
    n = len(r)

    if n == 0:
        return {
            "n_days": 0,
            "total_return": np.nan,
            "annual_return": np.nan,
            "annual_volatility": np.nan,
            "sharpe": np.nan,
            "sortino": np.nan,
            "max_drawdown": np.nan,
            "calmar": np.nan,
        }

    equity = (1.0 + r).cumprod()
    dd = equity / equity.cummax() - 1.0

    total_return = float(equity.iloc[-1] - 1.0)
    annual_return = float(equity.iloc[-1] ** (ANNUALIZATION_DAYS / n) - 1.0)

    daily_vol = float(r.std(ddof=1)) if n > 1 else np.nan
    annual_vol = daily_vol * np.sqrt(ANNUALIZATION_DAYS) if np.isfinite(daily_vol) else np.nan
    sharpe = float(r.mean() / daily_vol * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(daily_vol) and daily_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1)) if len(downside) > 1 else np.nan
    sortino = float(r.mean() / downside_vol * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(downside_vol) and downside_vol > 0 else np.nan

    max_dd = float(dd.min())
    calmar = float(annual_return / abs(max_dd)) if max_dd < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": str(r.index.min().date()) if isinstance(r.index, pd.DatetimeIndex) else "",
        "end_date": str(r.index.max().date()) if isinstance(r.index, pd.DatetimeIndex) else "",
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_dd,
        "calmar": calmar,
        "avg_daily_return": float(r.mean()),
        "daily_volatility": daily_vol,
        "hit_rate": float((r > 0).mean()),
    }

def drawdown_from_returns(r):
    r = pd.Series(r).dropna().astype(float)
    equity = (1.0 + r).cumprod()
    return equity / equity.cummax() - 1.0

def safe_corr(a, b):
    a = pd.Series(a).astype(float)
    b = pd.Series(b).astype(float)
    common = a.dropna().index.intersection(b.dropna().index)
    if len(common) < 3:
        return np.nan
    if a.loc[common].std(ddof=1) == 0 or b.loc[common].std(ddof=1) == 0:
        return np.nan
    return float(a.loc[common].corr(b.loc[common]))

def annualized_tracking_error(diff):
    diff = pd.Series(diff).dropna().astype(float)
    if len(diff) < 2:
        return np.nan
    return float(diff.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS))

# ============================================================
# 3. Strategy extraction utilities
# ============================================================

MAIN_AURORA_CANDIDATE_LABELS = [
    "AURORA10-UAMV-B",
    "AURORA10_UAMV_B",
    "AURORA10_UAMV_B_more60_defensive",
    "AURORA10-UAMV-B_more60_defensive",
    "UAMV_B_more60_defensive",
    "Primary defensive AURORA variant",
]

DIAG_AURORA_CANDIDATE_LABELS = [
    "Original dynamic AURORA",
    "AURORA18_DYNAMIC_ORIGINAL",
    "Dynamic AURORA",
    "Original_dynamic_AURORA",
]

RETURN_COL_CANDIDATES = [
    "net_return",
    "daily_return",
    "return",
    "returns",
    "strategy_return",
    "gross_return",
]

STRATEGY_COL_CANDIDATES = [
    "strategy_control",
    "strategy_name",
    "strategy",
    "policy_name",
    "control",
    "label",
    "portfolio",
]

def extract_series_from_long_or_wide(df, label_candidates, value_col_candidates=None, series_name="series"):
    if isinstance(label_candidates, str):
        label_candidates = [label_candidates]

    if value_col_candidates is None:
        value_col_candidates = RETURN_COL_CANDIDATES

    df = df.copy()

    # 1. Long format: strategy column + return column.
    for name_col in STRATEGY_COL_CANDIDATES:
        if name_col not in df.columns:
            continue

        ret_cols = [c for c in value_col_candidates if c in df.columns]
        if not ret_cols:
            continue

        ret_col = ret_cols[0]
        norm_names = df[name_col].astype(str).map(normalize_name)

        # Exact normalized label match.
        for lab in label_candidates:
            nlab = normalize_name(lab)
            mask = norm_names == nlab
            if mask.any():
                s = pd.to_numeric(df.loc[mask, ret_col], errors="coerce").dropna()
                s.index = pd.to_datetime(df.loc[mask].index)
                s.name = series_name
                return s.sort_index(), {
                    "format": "long",
                    "strategy_column": name_col,
                    "return_column": ret_col,
                    "matched_label": lab,
                    "match_type": "exact_normalized",
                }

        # Contains match.
        for lab in label_candidates:
            nlab = normalize_name(lab)
            mask = norm_names.map(lambda x: nlab in x or x in nlab)
            if mask.any():
                s = pd.to_numeric(df.loc[mask, ret_col], errors="coerce").dropna()
                s.index = pd.to_datetime(df.loc[mask].index)
                s.name = series_name
                return s.sort_index(), {
                    "format": "long",
                    "strategy_column": name_col,
                    "return_column": ret_col,
                    "matched_label": lab,
                    "match_type": "contains_normalized",
                }

    # 2. Wide format: each column is a strategy return series.
    norm_cols = {c: normalize_name(c) for c in df.columns}

    for lab in label_candidates:
        nlab = normalize_name(lab)
        for c, nc in norm_cols.items():
            if nc == nlab:
                s = pd.to_numeric(df[c], errors="coerce").dropna()
                s.name = series_name
                return s.sort_index(), {
                    "format": "wide",
                    "return_column": c,
                    "matched_label": lab,
                    "match_type": "exact_normalized_column",
                }

    for lab in label_candidates:
        nlab = normalize_name(lab)
        for c, nc in norm_cols.items():
            if nlab in nc or nc in nlab:
                s = pd.to_numeric(df[c], errors="coerce").dropna()
                s.name = series_name
                return s.sort_index(), {
                    "format": "wide",
                    "return_column": c,
                    "matched_label": lab,
                    "match_type": "contains_normalized_column",
                }

    # 3. If no match, show helpful error.
    raise ValueError(
        f"Could not extract {series_name}. "
        f"Labels tried: {label_candidates}. "
        f"Columns available: {list(df.columns)[:80]}"
    )

def standardize_weight_columns(wdf):
    wdf = wdf.copy()
    rename = {}
    for c in wdf.columns:
        nc = normalize_name(c)

        if "006208" in nc:
            rename[c] = "006208"
        elif "00692" in nc:
            rename[c] = "00692"
        elif "00881" in nc:
            rename[c] = "00881"
        elif "0050" in nc:
            rename[c] = "0050"
        elif "cash" in nc:
            rename[c] = "cash"

    wdf = wdf.rename(columns=rename)

    keep = [c for c in ["0050", "006208", "00692", "00881", "cash"] if c in wdf.columns]
    if not keep:
        return pd.DataFrame(index=wdf.index)

    out = wdf[keep].apply(pd.to_numeric, errors="coerce")
    out = out.groupby(level=0, axis=1).last()
    return out.sort_index()

def extract_weights_from_long_or_wide(df, label_candidates, series_name="weights"):
    if isinstance(label_candidates, str):
        label_candidates = [label_candidates]

    df = df.copy()

    # 1. Long format with strategy and weight columns.
    asset_cols = ["asset", "ticker", "symbol", "asset_name"]
    weight_cols = ["weight", "target_weight", "w", "portfolio_weight"]

    for name_col in STRATEGY_COL_CANDIDATES:
        if name_col not in df.columns:
            continue

        norm_names = df[name_col].astype(str).map(normalize_name)

        for lab in label_candidates:
            nlab = normalize_name(lab)
            mask = norm_names.map(lambda x: x == nlab or nlab in x or x in nlab)

            if not mask.any():
                continue

            sub = df.loc[mask].copy()
            sub.index = pd.to_datetime(sub.index)

            asset_col = next((c for c in asset_cols if c in sub.columns), None)
            weight_col = next((c for c in weight_cols if c in sub.columns), None)

            if asset_col and weight_col:
                wide = sub.pivot_table(index=sub.index, columns=asset_col, values=weight_col, aggfunc="last")
                out = standardize_weight_columns(wide)
                if not out.empty:
                    return out, {
                        "format": "long_asset_weight",
                        "strategy_column": name_col,
                        "asset_column": asset_col,
                        "weight_column": weight_col,
                        "matched_label": lab,
                    }

            possible_weight_cols = []
            for c in sub.columns:
                nc = normalize_name(c)
                if any(tok in nc for tok in ["0050", "006208", "00692", "00881", "cash"]):
                    possible_weight_cols.append(c)

            if possible_weight_cols:
                out = standardize_weight_columns(sub[possible_weight_cols])
                if not out.empty:
                    return out, {
                        "format": "long_strategy_wide_assets",
                        "strategy_column": name_col,
                        "matched_label": lab,
                        "asset_weight_columns": possible_weight_cols,
                    }

    # 2. Wide columns containing label and asset.
    selected = {}
    for lab in label_candidates:
        nlab = normalize_name(lab)
        for c in df.columns:
            nc = normalize_name(c)
            if nlab in nc:
                if "006208" in nc:
                    selected[c] = "006208"
                elif "00692" in nc:
                    selected[c] = "00692"
                elif "00881" in nc:
                    selected[c] = "00881"
                elif "0050" in nc:
                    selected[c] = "0050"
                elif "cash" in nc:
                    selected[c] = "cash"

    if selected:
        out = df[list(selected.keys())].copy()
        out.columns = [selected[c] for c in out.columns]
        out = standardize_weight_columns(out)
        if not out.empty:
            return out, {
                "format": "wide_label_asset_columns",
                "matched_columns": selected,
            }

    # 3. Generic wide weights without strategy labels.
    possible = {}
    for c in df.columns:
        nc = normalize_name(c)
        if "006208" in nc:
            possible[c] = "006208"
        elif "00692" in nc:
            possible[c] = "00692"
        elif "00881" in nc:
            possible[c] = "00881"
        elif "0050" in nc:
            possible[c] = "0050"
        elif "cash" in nc:
            possible[c] = "cash"

    if len(possible) >= 2:
        out = df[list(possible.keys())].copy()
        out.columns = [possible[c] for c in out.columns]
        out = standardize_weight_columns(out)
        if not out.empty:
            return out, {
                "format": "generic_wide_asset_columns",
                "matched_columns": possible,
                "warning": "No strategy label identified; using generic asset-weight columns.",
            }

    return None, {
        "format": "not_found",
        "reason": "Could not extract weight columns.",
        "columns": list(df.columns)[:80],
    }

def cash_series_from_weights(wdf):
    wdf = standardize_weight_columns(wdf)

    if "cash" in wdf.columns:
        return pd.to_numeric(wdf["cash"], errors="coerce").dropna()

    risky_cols = [c for c in ["0050", "006208", "00692", "00881"] if c in wdf.columns]
    if risky_cols:
        return (1.0 - wdf[risky_cols].sum(axis=1)).dropna()

    return pd.Series(dtype=float)

def turnover_from_weights(wdf):
    wdf = standardize_weight_columns(wdf).dropna(how="all")
    if wdf.empty:
        return pd.Series(dtype=float)
    turnover = wdf.diff().abs().sum(axis=1) / 2.0
    turnover.iloc[0] = wdf.iloc[0].abs().sum() / 2.0
    return turnover

# ============================================================
# 4. Locate input files
# ============================================================

search_roots = [OUTPUT_ROOT, PROJECT_ROOT]

# Main source-aware return matrix.
main_return_candidates = find_files(
    [
        "*source*aware*return*matrix*.parquet",
        "*source*aware*return*matrix*.csv",
        "*notebook13B*return*.parquet",
        "*notebook13B*return*.csv",
        "*NOTEBOOK13B*return*.parquet",
        "*NOTEBOOK13B*return*.csv",
        "*strict*test*return*matrix*.parquet",
        "*strict*test*return*matrix*.csv",
        "*aligned*return*matrix*.parquet",
        "*aligned*return*matrix*.csv",
    ],
    search_roots,
)

main_return_path = select_best_file(
    main_return_candidates,
    score_path_for_main_source_aware_return_matrix,
    "Candidate main source-aware return files",
    min_score=5,
)

if main_return_path is None:
    raise FileNotFoundError(
        "Could not find the main source-aware return matrix. "
        "Please provide its path manually in main_return_path."
    )

# Diagnostic Notebook 18 returns.
diag_return_candidates = find_files(
    ["all_notebook18_returns.parquet", "all_notebook18_returns.csv"],
    search_roots,
)

diag_return_path = select_best_file(
    diag_return_candidates,
    score_path_for_diagnostic_notebook18_returns,
    "Candidate Notebook 18 diagnostic return files",
    min_score=10,
)

if diag_return_path is None:
    raise FileNotFoundError(
        "Could not find Notebook 18 diagnostic returns. "
        "Expected all_notebook18_returns.parquet/csv."
    )

# Diagnostic Notebook 18 weights.
diag_weight_candidates = find_files(
    ["all_notebook18_weights.parquet", "all_notebook18_weights.csv"],
    search_roots,
)

diag_weight_path = select_best_file(
    diag_weight_candidates,
    score_path_for_diagnostic_notebook18_weights,
    "Candidate Notebook 18 diagnostic weight files",
    min_score=10,
)

# Main source-aware weights, if available.
main_weight_candidates = find_files(
    [
        "*source*aware*weight*.parquet",
        "*source*aware*weight*.csv",
        "*notebook13B*weight*.parquet",
        "*notebook13B*weight*.csv",
        "*AURORA*weight*.parquet",
        "*AURORA*weight*.csv",
        "*weights*.parquet",
        "*weights*.csv",
    ],
    search_roots,
    max_files=500,
)

main_weight_path = select_best_file(
    main_weight_candidates,
    score_path_for_main_weights,
    "Candidate main source-aware AURORA weight files",
    min_score=8,
)

# ============================================================
# 5. Load and extract main/diagnostic returns
# ============================================================

print("\n" + "=" * 100)
print("Loading and extracting return series")
print("=" * 100)

main_return_df = read_table_auto(main_return_path)
diag_return_df = read_table_auto(diag_return_path)

main_ret, main_ret_meta = extract_series_from_long_or_wide(
    main_return_df,
    MAIN_AURORA_CANDIDATE_LABELS,
    RETURN_COL_CANDIDATES,
    series_name="main_source_aware_AURORA",
)

diag_ret, diag_ret_meta = extract_series_from_long_or_wide(
    diag_return_df,
    DIAG_AURORA_CANDIDATE_LABELS,
    RETURN_COL_CANDIDATES,
    series_name="diagnostic_AURORA",
)

# Restrict to aligned evaluation window.
main_ret = main_ret.loc[(main_ret.index >= STRICT_START) & (main_ret.index <= STRICT_END)].dropna()
diag_ret = diag_ret.loc[(diag_ret.index >= STRICT_START) & (diag_ret.index <= STRICT_END)].dropna()

common_dates = main_ret.index.intersection(diag_ret.index).sort_values()

if len(common_dates) == 0:
    raise ValueError("No common dates between main and diagnostic AURORA return series.")

main_common = main_ret.loc[common_dates]
diag_common = diag_ret.loc[common_dates]

main_dd = drawdown_from_returns(main_common)
diag_dd = drawdown_from_returns(diag_common)

print("Main return extraction:", main_ret_meta)
print("Diagnostic return extraction:", diag_ret_meta)
print("Common dates:", len(common_dates), common_dates.min().date(), "to", common_dates.max().date())

# ============================================================
# 6. Load and extract weights, if available
# ============================================================

print("\n" + "=" * 100)
print("Loading and extracting weight series if available")
print("=" * 100)

main_weight_meta = None
diag_weight_meta = None
main_w = None
diag_w = None
main_cash = pd.Series(dtype=float)
diag_cash = pd.Series(dtype=float)
main_turnover = pd.Series(dtype=float)
diag_turnover = pd.Series(dtype=float)

if main_weight_path is not None:
    try:
        main_weight_df = read_table_auto(main_weight_path)
        main_w, main_weight_meta = extract_weights_from_long_or_wide(
            main_weight_df,
            MAIN_AURORA_CANDIDATE_LABELS,
            series_name="main_source_aware_AURORA_weights",
        )
        if main_w is not None and not main_w.empty:
            main_w = main_w.loc[(main_w.index >= STRICT_START) & (main_w.index <= STRICT_END)]
            main_cash = cash_series_from_weights(main_w)
            main_turnover = turnover_from_weights(main_w)
        print("Main weight extraction:", main_weight_meta)
    except Exception as e:
        main_weight_meta = {"error": repr(e)}
        print("Main weight extraction failed:", repr(e))
else:
    main_weight_meta = {"status": "not_found"}
    print("Main weight file not found. Cash-weight bridge will be unavailable for main source-aware series.")

if diag_weight_path is not None:
    try:
        diag_weight_df = read_table_auto(diag_weight_path)
        diag_w, diag_weight_meta = extract_weights_from_long_or_wide(
            diag_weight_df,
            DIAG_AURORA_CANDIDATE_LABELS,
            series_name="diagnostic_AURORA_weights",
        )
        if diag_w is not None and not diag_w.empty:
            diag_w = diag_w.loc[(diag_w.index >= STRICT_START) & (diag_w.index <= STRICT_END)]
            diag_cash = cash_series_from_weights(diag_w)
            diag_turnover = turnover_from_weights(diag_w)
        print("Diagnostic weight extraction:", diag_weight_meta)
    except Exception as e:
        diag_weight_meta = {"error": repr(e)}
        print("Diagnostic weight extraction failed:", repr(e))
else:
    diag_weight_meta = {"status": "not_found"}
    print("Diagnostic weight file not found.")

# Common dates for weights.
weight_common_dates = main_cash.index.intersection(diag_cash.index).sort_values()

# ============================================================
# 7. Compute bridge diagnostics
# ============================================================

print("\n" + "=" * 100)
print("Computing bridge diagnostics")
print("=" * 100)

main_perf = performance_metrics_from_returns(main_common)
diag_perf = performance_metrics_from_returns(diag_common)

return_diff = main_common - diag_common
abs_return_diff = return_diff.abs()

main_cum = (1.0 + main_common).cumprod() - 1.0
diag_cum = (1.0 + diag_common).cumprod() - 1.0
cum_diff = main_cum - diag_cum

bridge_daily = pd.DataFrame(index=common_dates)
bridge_daily.index.name = "date"
bridge_daily["main_source_aware_return"] = main_common
bridge_daily["diagnostic_return"] = diag_common
bridge_daily["return_difference_main_minus_diagnostic"] = return_diff
bridge_daily["abs_return_difference"] = abs_return_diff
bridge_daily["main_cumulative_return"] = main_cum
bridge_daily["diagnostic_cumulative_return"] = diag_cum
bridge_daily["cumulative_return_difference"] = cum_diff
bridge_daily["main_drawdown"] = main_dd
bridge_daily["diagnostic_drawdown"] = diag_dd
bridge_daily["drawdown_difference_main_minus_diagnostic"] = main_dd - diag_dd

if len(weight_common_dates) > 0:
    bridge_daily["main_cash_weight"] = main_cash.reindex(common_dates)
    bridge_daily["diagnostic_cash_weight"] = diag_cash.reindex(common_dates)
    bridge_daily["cash_weight_difference_main_minus_diagnostic"] = bridge_daily["main_cash_weight"] - bridge_daily["diagnostic_cash_weight"]
    bridge_daily["abs_cash_weight_difference"] = bridge_daily["cash_weight_difference_main_minus_diagnostic"].abs()
    bridge_daily["main_at_cash_cap"] = (bridge_daily["main_cash_weight"] >= CASH_CAP - 1e-8).astype(float)
    bridge_daily["diagnostic_at_cash_cap"] = (bridge_daily["diagnostic_cash_weight"] >= CASH_CAP - 1e-8).astype(float)

if len(main_turnover) > 0:
    bridge_daily["main_turnover_estimated_from_weights"] = main_turnover.reindex(common_dates)

if len(diag_turnover) > 0:
    bridge_daily["diagnostic_turnover_estimated_from_weights"] = diag_turnover.reindex(common_dates)

# Weight vector similarity if both weights are available.
weight_vector_stats = {}

if main_w is not None and diag_w is not None and not main_w.empty and not diag_w.empty:
    common_weight_dates = main_w.index.intersection(diag_w.index).intersection(common_dates).sort_values()
    main_w_std = standardize_weight_columns(main_w).reindex(common_weight_dates)
    diag_w_std = standardize_weight_columns(diag_w).reindex(common_weight_dates)
    common_weight_cols = [c for c in ["0050", "006208", "00692", "00881", "cash"] if c in main_w_std.columns and c in diag_w_std.columns]

    if len(common_weight_dates) > 0 and common_weight_cols:
        diff_w = main_w_std[common_weight_cols] - diag_w_std[common_weight_cols]
        abs_diff_w = diff_w.abs()

        identical_target_dates = (abs_diff_w.max(axis=1) <= 1e-8)

        weight_vector_stats = {
            "weight_common_dates": int(len(common_weight_dates)),
            "common_weight_columns": ",".join(common_weight_cols),
            "mean_abs_weight_difference_all_common_assets": float(abs_diff_w.mean(axis=1).mean()),
            "median_abs_weight_difference_all_common_assets": float(abs_diff_w.median(axis=1).median()),
            "max_abs_weight_difference_all_common_assets": float(abs_diff_w.max(axis=1).max()),
            "pct_dates_identical_target_weights_tolerance_1e_minus_8": float(identical_target_dates.mean()),
        }

        for col in common_weight_cols:
            weight_vector_stats[f"mean_abs_{col}_weight_difference"] = float(abs_diff_w[col].mean())
            weight_vector_stats[f"{col}_weight_correlation"] = safe_corr(main_w_std[col], diag_w_std[col])

# Scalar bridge metrics.
scalar_metrics = {
    "common_dates": int(len(common_dates)),
    "common_start_date": str(common_dates.min().date()),
    "common_end_date": str(common_dates.max().date()),
    "main_first_return_date_in_window": str(main_ret.index.min().date()) if len(main_ret) else "",
    "diagnostic_first_return_date_in_window": str(diag_ret.index.min().date()) if len(diag_ret) else "",
    "main_n_days_in_window": int(len(main_ret)),
    "diagnostic_n_days_in_window": int(len(diag_ret)),
    "daily_return_correlation": safe_corr(main_common, diag_common),
    "drawdown_correlation": safe_corr(main_dd, diag_dd),
    "mean_signed_daily_return_difference": float(return_diff.mean()),
    "mean_absolute_daily_return_difference": float(abs_return_diff.mean()),
    "median_absolute_daily_return_difference": float(abs_return_diff.median()),
    "root_mean_squared_daily_return_difference": float(np.sqrt(np.mean(return_diff.values ** 2))),
    "annualized_tracking_error_main_minus_diagnostic": annualized_tracking_error(return_diff),
    "final_cumulative_return_difference": float(cum_diff.iloc[-1]),
    "max_absolute_cumulative_return_difference": float(cum_diff.abs().max()),
    "main_total_return": main_perf["total_return"],
    "diagnostic_total_return": diag_perf["total_return"],
    "total_return_difference_main_minus_diagnostic": main_perf["total_return"] - diag_perf["total_return"],
    "main_sharpe": main_perf["sharpe"],
    "diagnostic_sharpe": diag_perf["sharpe"],
    "sharpe_difference_main_minus_diagnostic": main_perf["sharpe"] - diag_perf["sharpe"],
    "main_sortino": main_perf["sortino"],
    "diagnostic_sortino": diag_perf["sortino"],
    "sortino_difference_main_minus_diagnostic": main_perf["sortino"] - diag_perf["sortino"],
    "main_max_drawdown": main_perf["max_drawdown"],
    "diagnostic_max_drawdown": diag_perf["max_drawdown"],
    "max_drawdown_difference_main_minus_diagnostic": main_perf["max_drawdown"] - diag_perf["max_drawdown"],
    "main_annual_volatility": main_perf["annual_volatility"],
    "diagnostic_annual_volatility": diag_perf["annual_volatility"],
}

if len(weight_common_dates) > 0:
    scalar_metrics.update({
        "cash_common_dates": int(len(weight_common_dates)),
        "main_avg_cash_weight": float(main_cash.reindex(weight_common_dates).mean()),
        "diagnostic_avg_cash_weight": float(diag_cash.reindex(weight_common_dates).mean()),
        "avg_cash_weight_difference_main_minus_diagnostic": float(
            main_cash.reindex(weight_common_dates).mean() - diag_cash.reindex(weight_common_dates).mean()
        ),
        "cash_weight_correlation": safe_corr(main_cash.reindex(weight_common_dates), diag_cash.reindex(weight_common_dates)),
        "mean_absolute_cash_weight_difference": float(
            (main_cash.reindex(weight_common_dates) - diag_cash.reindex(weight_common_dates)).abs().mean()
        ),
        "main_cash_cap_binding_frequency": float((main_cash.reindex(weight_common_dates) >= CASH_CAP - 1e-8).mean()),
        "diagnostic_cash_cap_binding_frequency": float((diag_cash.reindex(weight_common_dates) >= CASH_CAP - 1e-8).mean()),
        "cash_cap_binding_frequency_difference_main_minus_diagnostic": float(
            (main_cash.reindex(weight_common_dates) >= CASH_CAP - 1e-8).mean()
            - (diag_cash.reindex(weight_common_dates) >= CASH_CAP - 1e-8).mean()
        ),
    })
else:
    scalar_metrics.update({
        "cash_common_dates": 0,
        "main_avg_cash_weight": np.nan,
        "diagnostic_avg_cash_weight": np.nan,
        "avg_cash_weight_difference_main_minus_diagnostic": np.nan,
        "cash_weight_correlation": np.nan,
        "mean_absolute_cash_weight_difference": np.nan,
        "main_cash_cap_binding_frequency": np.nan,
        "diagnostic_cash_cap_binding_frequency": np.nan,
        "cash_cap_binding_frequency_difference_main_minus_diagnostic": np.nan,
    })

if len(main_turnover) > 0 and len(diag_turnover) > 0:
    common_turnover_dates = main_turnover.index.intersection(diag_turnover.index).intersection(common_dates)
    scalar_metrics.update({
        "turnover_common_dates": int(len(common_turnover_dates)),
        "main_total_turnover_estimated_from_weights": float(main_turnover.reindex(common_turnover_dates).sum()),
        "diagnostic_total_turnover_estimated_from_weights": float(diag_turnover.reindex(common_turnover_dates).sum()),
        "total_turnover_difference_main_minus_diagnostic": float(
            main_turnover.reindex(common_turnover_dates).sum() - diag_turnover.reindex(common_turnover_dates).sum()
        ),
        "main_avg_turnover_estimated_from_weights": float(main_turnover.reindex(common_turnover_dates).mean()),
        "diagnostic_avg_turnover_estimated_from_weights": float(diag_turnover.reindex(common_turnover_dates).mean()),
    })
else:
    scalar_metrics.update({
        "turnover_common_dates": 0,
        "main_total_turnover_estimated_from_weights": np.nan,
        "diagnostic_total_turnover_estimated_from_weights": np.nan,
        "total_turnover_difference_main_minus_diagnostic": np.nan,
        "main_avg_turnover_estimated_from_weights": np.nan,
        "diagnostic_avg_turnover_estimated_from_weights": np.nan,
    })

scalar_metrics.update(weight_vector_stats)

# Interpretation thresholds.
return_corr = scalar_metrics["daily_return_correlation"]
cash_corr = scalar_metrics["cash_weight_correlation"]
mean_abs_ret_diff = scalar_metrics["mean_absolute_daily_return_difference"]
avg_cash_diff = scalar_metrics["avg_cash_weight_difference_main_minus_diagnostic"]

if np.isfinite(return_corr) and return_corr >= 0.90:
    return_alignment = "High daily-return alignment"
elif np.isfinite(return_corr) and return_corr >= 0.70:
    return_alignment = "Moderate daily-return alignment"
elif np.isfinite(return_corr):
    return_alignment = "Weak daily-return alignment"
else:
    return_alignment = "Daily-return alignment unavailable"

if np.isfinite(cash_corr) and cash_corr >= 0.90 and np.isfinite(avg_cash_diff) and abs(avg_cash_diff) <= 0.05:
    exposure_alignment = "High cash-exposure alignment"
elif np.isfinite(cash_corr):
    exposure_alignment = "Cash-exposure alignment available but not high"
else:
    exposure_alignment = "Cash-exposure alignment unavailable"

if return_alignment.startswith("High") and exposure_alignment.startswith("High"):
    recommended_claim_scope = (
        "Bridge diagnostics support using the diagnostic reconstruction as a close mechanism-attribution proxy "
        "for the main source-aware AURORA behavior."
    )
elif return_alignment.startswith("High") and exposure_alignment.endswith("unavailable"):
    recommended_claim_scope = (
        "Bridge diagnostics support high return alignment, but cash-weight alignment is unavailable; "
        "mechanism claims should remain qualified by pipeline differences."
    )
else:
    recommended_claim_scope = (
        "Bridge diagnostics do not establish close equivalence between the main source-aware series and the diagnostic reconstruction; "
        "constant-lambda and no-probability controls should be interpreted as applying to the standalone AURORA-compatible diagnostic reconstruction."
    )

# ============================================================
# 8. Build Table S32
# ============================================================

rows = []

def add_row(metric, value, interpretation, category="bridge"):
    rows.append({
        "category": category,
        "metric": metric,
        "value": value,
        "interpretation": interpretation,
    })

add_row("Common dates", scalar_metrics["common_dates"], "Number of overlapping daily observations used in bridge comparison.", "sample")
add_row("Common date range", f"{scalar_metrics['common_start_date']} to {scalar_metrics['common_end_date']}", "Overlapping aligned evaluation date range.", "sample")
add_row("Main first return date in window", scalar_metrics["main_first_return_date_in_window"], "First main source-aware AURORA return date found in evaluation window.", "sample")
add_row("Diagnostic first return date in window", scalar_metrics["diagnostic_first_return_date_in_window"], "First diagnostic AURORA return date found in evaluation window.", "sample")

add_row("Daily return correlation", scalar_metrics["daily_return_correlation"], return_alignment, "returns")
add_row("Drawdown correlation", scalar_metrics["drawdown_correlation"], "Correlation between drawdown paths computed on common dates.", "returns")
add_row("Mean absolute daily return difference", scalar_metrics["mean_absolute_daily_return_difference"], "Average absolute daily return gap between main and diagnostic AURORA.", "returns")
add_row("Median absolute daily return difference", scalar_metrics["median_absolute_daily_return_difference"], "Median absolute daily return gap between main and diagnostic AURORA.", "returns")
add_row("Annualized tracking error", scalar_metrics["annualized_tracking_error_main_minus_diagnostic"], "Annualized standard deviation of daily return difference.", "returns")
add_row("Final cumulative return difference", scalar_metrics["final_cumulative_return_difference"], "Final cumulative return gap: main minus diagnostic.", "returns")
add_row("Maximum absolute cumulative return difference", scalar_metrics["max_absolute_cumulative_return_difference"], "Largest absolute cumulative return gap over common dates.", "returns")

add_row("Main total return", scalar_metrics["main_total_return"], "Performance of main source-aware AURORA series on common dates.", "performance")
add_row("Diagnostic total return", scalar_metrics["diagnostic_total_return"], "Performance of diagnostic AURORA reconstruction on common dates.", "performance")
add_row("Total return difference", scalar_metrics["total_return_difference_main_minus_diagnostic"], "Main minus diagnostic total return.", "performance")
add_row("Main Sharpe", scalar_metrics["main_sharpe"], "Sharpe ratio of main source-aware AURORA series on common dates.", "performance")
add_row("Diagnostic Sharpe", scalar_metrics["diagnostic_sharpe"], "Sharpe ratio of diagnostic AURORA reconstruction on common dates.", "performance")
add_row("Main Sortino", scalar_metrics["main_sortino"], "Sortino ratio of main source-aware AURORA series on common dates.", "performance")
add_row("Diagnostic Sortino", scalar_metrics["diagnostic_sortino"], "Sortino ratio of diagnostic AURORA reconstruction on common dates.", "performance")
add_row("Main max drawdown", scalar_metrics["main_max_drawdown"], "Maximum drawdown of main source-aware AURORA series on common dates.", "performance")
add_row("Diagnostic max drawdown", scalar_metrics["diagnostic_max_drawdown"], "Maximum drawdown of diagnostic AURORA reconstruction on common dates.", "performance")

add_row("Cash common dates", scalar_metrics["cash_common_dates"], "Number of dates with both main and diagnostic cash weights available.", "exposure")
add_row("Main average cash weight", scalar_metrics["main_avg_cash_weight"], "Average cash weight of main source-aware AURORA if available.", "exposure")
add_row("Diagnostic average cash weight", scalar_metrics["diagnostic_avg_cash_weight"], "Average cash weight of diagnostic AURORA reconstruction.", "exposure")
add_row("Average cash-weight difference", scalar_metrics["avg_cash_weight_difference_main_minus_diagnostic"], "Main minus diagnostic average cash weight.", "exposure")
add_row("Cash-weight correlation", scalar_metrics["cash_weight_correlation"], exposure_alignment, "exposure")
add_row("Mean absolute cash-weight difference", scalar_metrics["mean_absolute_cash_weight_difference"], "Average absolute cash-weight gap if both weight series are available.", "exposure")
add_row("Main cash-cap binding frequency", scalar_metrics["main_cash_cap_binding_frequency"], "Frequency at which main AURORA cash weight is at 0.60 cap if available.", "exposure")
add_row("Diagnostic cash-cap binding frequency", scalar_metrics["diagnostic_cash_cap_binding_frequency"], "Frequency at which diagnostic AURORA cash weight is at 0.60 cap.", "exposure")

if weight_vector_stats:
    for k, v in weight_vector_stats.items():
        add_row(k, v, "Weight-vector bridge diagnostic.", "weights")

add_row("Recommended claim scope", recommended_claim_scope, "Use this row to choose the final manuscript wording.", "interpretation")

table_s32 = pd.DataFrame(rows)

# Rounded version.
table_s32_rounded = table_s32.copy()
for idx, row in table_s32_rounded.iterrows():
    val = row["value"]
    if isinstance(val, (int, np.integer)):
        continue
    try:
        fval = float(val)
        if np.isfinite(fval):
            table_s32_rounded.at[idx, "value"] = round(fval, 6)
    except Exception:
        pass

# ============================================================
# 9. Save outputs
# ============================================================

table_s32_path = RUN_TABLE_DIR / "table_S32_main_vs_diagnostic_AURORA_bridge.csv"
table_s32_rounded_path = RUN_TABLE_DIR / "table_S32_main_vs_diagnostic_AURORA_bridge_rounded.csv"
daily_path = RUN_DIAG_DIR / "main_vs_diagnostic_AURORA_bridge_daily.csv"
scalar_path = RUN_DIAG_DIR / "main_vs_diagnostic_AURORA_bridge_scalar_metrics.json"

table_s32.to_csv(table_s32_path, index=False)
table_s32_rounded.to_csv(table_s32_rounded_path, index=False)
bridge_daily.to_csv(daily_path)
save_json(scalar_path, scalar_metrics)

# Global copies.
global_table_s32_path = TABLE_DIR / "table_S32_main_vs_diagnostic_AURORA_bridge.csv"
global_table_s32_rounded_path = TABLE_DIR / "table_S32_main_vs_diagnostic_AURORA_bridge_rounded.csv"
global_daily_path = DIAG_DIR / "main_vs_diagnostic_AURORA_bridge_daily.csv"
global_scalar_path = DIAG_DIR / "main_vs_diagnostic_AURORA_bridge_scalar_metrics.json"

table_s32.to_csv(global_table_s32_path, index=False)
table_s32_rounded.to_csv(global_table_s32_rounded_path, index=False)
bridge_daily.to_csv(global_daily_path)
save_json(global_scalar_path, scalar_metrics)

# Recommended manuscript wording.
if recommended_claim_scope.startswith("Bridge diagnostics support using"):
    recommended_text = """
The main source-aware AURORA series and the standalone diagnostic AURORA reconstruction are not numerically identical because they are produced by different return-generation pipelines. However, the bridge diagnostic in Supplementary Table S32 shows close daily-return and cash-exposure alignment. Therefore, the attribution diagnostics are interpreted as evidence for the dominant cash-exposure and risk-aversion mechanism underlying the reported AURORA behavior, while the main source-aware series remains the authoritative series for AURORA-versus-ROMA comparison.
""".strip()
elif "high return alignment" in recommended_claim_scope.lower():
    recommended_text = """
The main source-aware AURORA series and the standalone diagnostic AURORA reconstruction are not numerically identical because they are produced by different return-generation pipelines. The bridge diagnostic in Supplementary Table S32 shows high daily-return alignment, but cash-exposure alignment is unavailable in the main source-aware weights. Therefore, the attribution diagnostics are interpreted as mechanism evidence for an AURORA-compatible diagnostic reconstruction and as supportive, but not exact, evidence for the main source-aware AURORA series.
""".strip()
else:
    recommended_text = """
The main source-aware AURORA series and the standalone diagnostic AURORA reconstruction are not numerically identical. The bridge diagnostic in Supplementary Table S32 does not establish close equivalence between the two series. Therefore, the constant-\\(\\lambda\\), no-probability, and shuffled-probability controls are interpreted as mechanism-attribution diagnostics for a standalone AURORA-compatible reconstruction rather than exact decompositions of the main source-aware return series. The main source-aware comparison remains valid as an AURORA-versus-ROMA comparison, but the mechanism conclusion is stated as evidence that the AURORA-compatible diagnostic reconstruction is primarily cash- and risk-aversion-driven.
""".strip()

recommended_text_path = RUN_REPORT_DIR / "recommended_manuscript_bridge_wording.txt"
global_recommended_text_path = REPORT_DIR / "recommended_manuscript_bridge_wording.txt"
recommended_text_path.write_text(recommended_text, encoding="utf-8")
global_recommended_text_path.write_text(recommended_text, encoding="utf-8")

# Validation report.
validation = {
    "project_code": PROJECT_CODE,
    "notebook": "24_bridge_main_source_aware_and_diagnostic_AURORA",
    "run_id": RUN_ID,
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "purpose": "Bridge main source-aware AURORA return series to standalone diagnostic AURORA reconstruction.",
    "input_files": {
        "main_source_aware_return_path": str(main_return_path),
        "diagnostic_return_path": str(diag_return_path),
        "main_weight_path": str(main_weight_path) if main_weight_path else None,
        "diagnostic_weight_path": str(diag_weight_path) if diag_weight_path else None,
    },
    "extraction_metadata": {
        "main_return": main_ret_meta,
        "diagnostic_return": diag_ret_meta,
        "main_weight": main_weight_meta,
        "diagnostic_weight": diag_weight_meta,
    },
    "evaluation_window": {
        "strict_start": str(STRICT_START.date()),
        "strict_end": str(STRICT_END.date()),
        "common_dates": int(len(common_dates)),
        "common_start": str(common_dates.min().date()),
        "common_end": str(common_dates.max().date()),
    },
    "key_scalar_metrics": scalar_metrics,
    "recommended_claim_scope": recommended_claim_scope,
    "outputs": {
        "table_s32": str(table_s32_path),
        "table_s32_rounded": str(table_s32_rounded_path),
        "daily_bridge": str(daily_path),
        "scalar_metrics": str(scalar_path),
        "recommended_wording": str(recommended_text_path),
        "global_table_s32": str(global_table_s32_path),
        "global_table_s32_rounded": str(global_table_s32_rounded_path),
        "global_daily_bridge": str(global_daily_path),
    },
}

validation_path = RUN_REPORT_DIR / "NOTEBOOK24_bridge_validation_report.json"
global_validation_path = REPORT_DIR / f"NOTEBOOK24_bridge_validation_report_{RUN_ID}.json"

save_json(validation_path, validation)
save_json(global_validation_path, validation)

# File manifest for this run.
manifest_rows = []
for p in sorted(RUN_ROOT.rglob("*")):
    if p.is_file():
        stat = p.stat()
        manifest_rows.append({
            "path": p.relative_to(RUN_ROOT).as_posix(),
            "size_bytes": int(stat.st_size),
            "modified_utc": datetime.fromtimestamp(stat.st_mtime, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "sha256": sha256_file(p),
        })

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = RUN_REPORT_DIR / "NOTEBOOK24_file_manifest_SHA256.csv"
global_manifest_path = REPORT_DIR / f"NOTEBOOK24_file_manifest_SHA256_{RUN_ID}.csv"
manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(global_manifest_path, index=False)

# ============================================================
# 10. Final summary
# ============================================================

print("\n" + "=" * 100)
print("NOTEBOOK 24 COMPLETE")
print("=" * 100)
print("Run root:", RUN_ROOT)
print("Table S32:", table_s32_path)
print("Table S32 rounded:", table_s32_rounded_path)
print("Daily bridge:", daily_path)
print("Validation report:", validation_path)
print("Global Table S32:", global_table_s32_path)
print("Global daily bridge:", global_daily_path)
print("=" * 100)

print("\nMain input files:")
print("Main source-aware returns:", main_return_path)
print("Diagnostic returns:", diag_return_path)
print("Main source-aware weights:", main_weight_path)
print("Diagnostic weights:", diag_weight_path)

print("\nKey bridge metrics:")
for k in [
    "common_dates",
    "daily_return_correlation",
    "drawdown_correlation",
    "mean_absolute_daily_return_difference",
    "annualized_tracking_error_main_minus_diagnostic",
    "final_cumulative_return_difference",
    "main_total_return",
    "diagnostic_total_return",
    "main_sharpe",
    "diagnostic_sharpe",
    "main_max_drawdown",
    "diagnostic_max_drawdown",
    "cash_common_dates",
    "main_avg_cash_weight",
    "diagnostic_avg_cash_weight",
    "cash_weight_correlation",
    "mean_absolute_cash_weight_difference",
    "main_cash_cap_binding_frequency",
    "diagnostic_cash_cap_binding_frequency",
]:
    print(f"{k}: {scalar_metrics.get(k)}")

print("\nRecommended claim scope:")
print(recommended_claim_scope)

print("\nRecommended manuscript wording:")
print(recommended_text)

print("\nTable S32 preview:")
display(table_s32_rounded)

print("=" * 100)

Mounted at /content/drive
Notebook 24: bridge main source-aware AURORA vs diagnostic AURORA
RUN_ID: 20260725_003004
PROJECT_ROOT: /content/drive/MyDrive/AURORA_TWETF
OUTPUT_ROOT: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF

----------------------------------------------------------------------------------------------------
Candidate main source-aware return files
----------------------------------------------------------------------------------------------------
  32  /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_AURORA_TWETF/source_aware_unified_paper_comparison/run_20260625_065916/returns/notebook13B_source_aware_strict_test_return_matrix.parquet
  32  /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_AURORA_TWETF/source_aware_unified_paper_comparison/run_20260625_065916/returns/notebook13B_source_aware_strict_test_return_matrix.csv
  20  /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/tables/table_S29c_source_aware_return_matrix_column_inventory_20260723_0355

,category,metric,value,interpretation
0,sample,Common dates,319,Number of overlapping daily observations used ...
1,sample,Common date range,2024-11-27 to 2026-03-25,Overlapping aligned evaluation date range.
2,sample,Main first return date in window,2024-11-27,First main source-aware AURORA return date fou...
3,sample,Diagnostic first return date in window,2024-11-27,First diagnostic AURORA return date found in e...
4,returns,Daily return correlation,0.967014,High daily-return alignment
5,returns,Drawdown correlation,0.988301,Correlation between drawdown paths computed on...
6,returns,Mean absolute daily return difference,0.000821,Average absolute daily return gap between main...
7,returns,Median absolute daily return difference,0.0002,Median absolute daily return gap between main ...
8,returns,Annualized tracking error,0.030608,Annualized standard deviation of daily return ...
9,returns,Final cumulative return difference,0.139236,Final cumulative return gap: main minus diagno...
